In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.cm as cm

path = "data/UNSEEN/SAPM-MA-03214.obj"

# --- Load mesh ---
mesh = o3d.io.read_triangle_mesh(path)
if mesh.is_empty():
    raise RuntimeError("Mesh could not be loaded.")

mesh.compute_vertex_normals()

# --- Optional decimation ---
TARGET_TRIANGLES = 100_000
if len(mesh.triangles) > TARGET_TRIANGLES:
    mesh = mesh.simplify_quadric_decimation(TARGET_TRIANGLES)
    mesh.compute_vertex_normals()

# --- Compute scalar (example: distance to centroid) ---
V = np.asarray(mesh.vertices)
centroid = V.mean(axis=0)
s = np.linalg.norm(V - centroid, axis=1)
s = (s - s.min()) / (s.max() - s.min() + 1e-12)

# --- Use matplotlib colormap ---
cmap = cm.get_cmap("viridis")  # try "inferno", "jet", etc.
colors = cmap(s)[:, :3]        # remove alpha channel

mesh.vertex_colors = o3d.utility.Vector3dVector(colors)

# --- Interactive Viewer with grey background ---
vis = o3d.visualization.Visualizer()
vis.create_window("Interactive Heatmap", width=1100, height=800)
vis.add_geometry(mesh)

opt = vis.get_render_option()
opt.background_color = np.array([0.85, 0.85, 0.85])  # light grey
opt.mesh_show_back_face = True

vis.run()
vis.destroy_window()